# A股市场动量因子构造（优化版）

本notebook基于开源证券研报《A股市场中如何构造动量因子？》，使用免费数据源（tushare、akshare）重新实现，并优化代码结构。

## 主要改进
1. 使用tushare和akshare替代jqdata
2. 使用pandas、numpy等高效库优化计算
3. 改进代码结构，提高可读性
4. 添加详细注释和错误处理

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import tushare as ts
import akshare as ak
import warnings
from typing import List, Dict, Tuple, Union
from datetime import datetime, timedelta
import scipy.stats as stats
from tqdm import tqdm

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 设置tushare token
ts.set_token('ce9f8c37a4af987f6328321ed4b3f9379f695d6d6bdc5b59d454e3ad')
pro = ts.pro_api()

## 1. 数据获取模块

使用tushare和akshare替代jqdata获取数据

In [ ]:
class DataProvider:
    """数据提供者类，封装tushare和akshare的数据获取功能"""
    
    def __init__(self):
        self.pro = pro
        
    def get_trade_cal(self, start_date: str, end_date: str) -> pd.DataFrame:
        """获取交易日历"""
        try:
            cal = self.pro.trade_cal(exchange='SSE', 
                                   start_date=start_date.replace('-', ''),
                                   end_date=end_date.replace('-', ''))
            return cal[cal['is_open'] == 1]['cal_date'].tolist()
        except Exception as e:
            print(f"获取交易日历失败: {e}")
            return []
    
    def get_stock_list(self, date: str = None) -> List[str]:
        """获取股票列表（A股全市场）"""
        try:
            # 获取A股股票列表
            stocks = self.pro.stock_basic(exchange='', 
                                         list_status='L',
                                         fields='ts_code,symbol,name,area,industry,list_date')
            
            # 过滤掉ST股票和科创板
            stocks = stocks[~stocks['name'].str.contains('ST|退', na=False)]
            stocks = stocks[~stocks['ts_code'].str.startswith('688')]
            
            return stocks['ts_code'].tolist()
        except Exception as e:
            print(f"获取股票列表失败: {e}")
            return []
    
    def get_price_data(self, ts_codes: List[str], start_date: str, end_date: str) -> pd.DataFrame:
        """获取股票价格数据"""
        all_data = []
        
        for ts_code in tqdm(ts_codes, desc="获取价格数据"):
            try:
                df = self.pro.daily(ts_code=ts_code,
                                  start_date=start_date.replace('-', ''),
                                  end_date=end_date.replace('-', ''),
                                  fields='ts_code,trade_date,open,high,low,close,vol,amount')
                
                if not df.empty:
                    df['trade_date'] = pd.to_datetime(df['trade_date'])
                    df = df.sort_values('trade_date')
                    all_data.append(df)
                    
            except Exception as e:
                print(f"获取{ts_code}数据失败: {e}")
                continue
        
        if all_data:
            result = pd.concat(all_data, ignore_index=True)
            return result.pivot_table(index='trade_date', columns='ts_code', values='close')
        else:
            return pd.DataFrame()

# 初始化数据提供者
data_provider = DataProvider()

## 2. 因子计算模块

实现动量因子的计算逻辑

In [ ]:
class MomentumFactorCalculator:
    """动量因子计算器"""
    
    def __init__(self, price_data: pd.DataFrame):
        self.price_data = price_data
        self.returns = price_data.pct_change()
        
    def calculate_amplitude(self, window: int = 20) -> pd.DataFrame:
        """计算振幅"""
        high_low_ratio = self.price_data.rolling(window).max() / self.price_data.rolling(window).min() - 1
        return high_low_ratio
    
    def calculate_momentum_factor_A(self, window: int = 120, lambda_ratio: float = 0.3) -> pd.DataFrame:
        """计算A类动量因子（基于振幅切割）
        
        Args:
            window: 回看窗口期
            lambda_ratio: 切割比例
        """
        momentum_factors = pd.DataFrame(index=self.returns.index, columns=self.returns.columns)
        
        for date in tqdm(self.returns.index[window:], desc="计算动量因子A"):
            # 获取窗口期内的数据
            window_returns = self.returns.loc[:date].tail(window)
            window_amplitude = self.calculate_amplitude(1).loc[:date].tail(window)
            
            for stock in self.returns.columns:
                if stock in window_returns.columns and stock in window_amplitude.columns:
                    stock_returns = window_returns[stock].dropna()
                    stock_amplitude = window_amplitude[stock].dropna()
                    
                    if len(stock_returns) >= window * 0.8:  # 至少80%的数据
                        # 按振幅排序，选择振幅较低的部分
                        combined_data = pd.DataFrame({
                            'returns': stock_returns,
                            'amplitude': stock_amplitude
                        }).dropna()
                        
                        if len(combined_data) > 0:
                            # 按振幅排序
                            sorted_data = combined_data.sort_values('amplitude')
                            # 选择振幅较低的lambda_ratio比例的数据
                            cutoff = int(len(sorted_data) * lambda_ratio)
                            selected_returns = sorted_data.head(cutoff)['returns']
                            
                            # 计算动量因子值（累积收益率）
                            momentum_factors.loc[date, stock] = selected_returns.sum()
        
        return momentum_factors
    
    def calculate_momentum_factor_B(self, window: int = 120, sort_method: str = 'sort') -> pd.DataFrame:
        """计算B类动量因子（基于时间维度切割）
        
        Args:
            window: 回看窗口期
            sort_method: 排序方法
        """
        momentum_factors = pd.DataFrame(index=self.returns.index, columns=self.returns.columns)
        
        for date in tqdm(self.returns.index[window:], desc="计算动量因子B"):
            # 获取窗口期内的数据
            window_returns = self.returns.loc[:date].tail(window)
            
            for stock in self.returns.columns:
                if stock in window_returns.columns:
                    stock_returns = window_returns[stock].dropna()
                    
                    if len(stock_returns) >= window * 0.8:
                        # 简单的累积收益率
                        momentum_factors.loc[date, stock] = stock_returns.sum()
        
        return momentum_factors

# 示例：创建因子计算器
# calculator = MomentumFactorCalculator(price_data)

## 3. 因子分析模块

实现IC分析、分组回测等功能

In [ ]:
class FactorAnalyzer:
    """因子分析器"""
    
    def __init__(self, factor_data: pd.DataFrame, price_data: pd.DataFrame):
        self.factor_data = factor_data
        self.price_data = price_data
        self.returns = price_data.pct_change().shift(-1)  # 下期收益率
        
    def calculate_ic(self) -> pd.Series:
        """计算IC值"""
        ic_series = []
        
        for date in self.factor_data.index:
            if date in self.returns.index:
                factor_values = self.factor_data.loc[date].dropna()
                return_values = self.returns.loc[date].dropna()
                
                # 找到共同的股票
                common_stocks = factor_values.index.intersection(return_values.index)
                
                if len(common_stocks) > 10:  # 至少10只股票
                    factor_aligned = factor_values[common_stocks]
                    return_aligned = return_values[common_stocks]
                    
                    # 计算Spearman相关系数
                    ic, _ = stats.spearmanr(factor_aligned, return_aligned)
                    ic_series.append(ic)
                else:
                    ic_series.append(np.nan)
            else:
                ic_series.append(np.nan)
        
        return pd.Series(ic_series, index=self.factor_data.index)
    
    def group_backtest(self, n_groups: int = 5) -> pd.DataFrame:
        """分组回测"""
        group_returns = {f'G{i+1}': [] for i in range(n_groups)}
        group_returns['long_short'] = []
        
        for date in self.factor_data.index:
            if date in self.returns.index:
                factor_values = self.factor_data.loc[date].dropna()
                return_values = self.returns.loc[date].dropna()
                
                # 找到共同的股票
                common_stocks = factor_values.index.intersection(return_values.index)
                
                if len(common_stocks) > n_groups * 5:  # 确保每组至少5只股票
                    factor_aligned = factor_values[common_stocks]
                    return_aligned = return_values[common_stocks]
                    
                    # 按因子值分组
                    groups = pd.qcut(factor_aligned, n_groups, labels=False, duplicates='drop')
                    
                    # 计算各组收益率
                    for i in range(n_groups):
                        group_stocks = groups[groups == i].index
                        if len(group_stocks) > 0:
                            group_ret = return_aligned[group_stocks].mean()
                            group_returns[f'G{i+1}'].append(group_ret)
                        else:
                            group_returns[f'G{i+1}'].append(np.nan)
                    
                    # 计算多空收益
                    if not np.isnan(group_returns[f'G{n_groups}'][-1]) and not np.isnan(group_returns['G1'][-1]):
                        long_short = group_returns[f'G{n_groups}'][-1] - group_returns['G1'][-1]
                        group_returns['long_short'].append(long_short)
                    else:
                        group_returns['long_short'].append(np.nan)
                else:
                    for key in group_returns.keys():
                        group_returns[key].append(np.nan)
            else:
                for key in group_returns.keys():
                    group_returns[key].append(np.nan)
        
        return pd.DataFrame(group_returns, index=self.factor_data.index)
    
    def plot_ic_analysis(self, ic_series: pd.Series):
        """绘制IC分析图"""
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
        
        # IC时序图
        ic_series.plot(ax=ax1, title='IC时序图')
        ax1.axhline(0, color='red', linestyle='--')
        ax1.grid(True)
        
        # IC分布直方图
        ic_series.dropna().hist(bins=30, ax=ax2, alpha=0.7)
        ax2.set_title('IC分布直方图')
        ax2.axvline(ic_series.mean(), color='red', linestyle='--', label=f'均值: {ic_series.mean():.4f}')
        ax2.legend()
        ax2.grid(True)
        
        plt.tight_layout()
        plt.show()
        
        # 打印统计信息
        print(f"IC均值: {ic_series.mean():.4f}")
        print(f"IC标准差: {ic_series.std():.4f}")
        print(f"ICIR: {ic_series.mean() / ic_series.std():.4f}")
        print(f"IC胜率: {(ic_series > 0).mean():.4f}")

# 示例：创建因子分析器
# analyzer = FactorAnalyzer(factor_data, price_data)

## 4. 主要执行流程

整合上述模块，执行完整的因子构造和分析流程

In [ ]:
def main_analysis(start_date: str = '2018-01-01', end_date: str = '2023-12-31'):
    """主要分析流程"""
    
    print("=== A股动量因子构造分析 ===")
    print(f"分析期间: {start_date} 至 {end_date}")
    
    # 1. 获取数据
    print("\n1. 获取基础数据...")
    stock_list = data_provider.get_stock_list()
    print(f"获取到{len(stock_list)}只股票")
    
    # 为了演示，只选择前100只股票
    stock_list = stock_list[:100]
    print(f"为演示目的，选择前{len(stock_list)}只股票")
    
    price_data = data_provider.get_price_data(stock_list, start_date, end_date)
    print(f"价格数据形状: {price_data.shape}")
    
    if price_data.empty:
        print("未获取到价格数据，程序退出")
        return
    
    # 2. 计算因子
    print("\n2. 计算动量因子...")
    calculator = MomentumFactorCalculator(price_data)
    
    # 计算A类因子（不同lambda值）
    lambda_values = [0.1, 0.2, 0.3, 0.4, 0.5]
    factor_results = {}
    
    for lambda_val in lambda_values:
        print(f"计算lambda={lambda_val}的A类因子...")
        factor_A = calculator.calculate_momentum_factor_A(window=60, lambda_ratio=lambda_val)
        factor_results[f'A_{lambda_val}'] = factor_A
    
    # 3. 因子分析
    print("\n3. 进行因子分析...")
    ic_results = {}
    
    for factor_name, factor_data in factor_results.items():
        print(f"分析因子: {factor_name}")
        analyzer = FactorAnalyzer(factor_data, price_data)
        ic_series = analyzer.calculate_ic()
        ic_results[factor_name] = {
            'ic_mean': ic_series.mean(),
            'ic_std': ic_series.std(),
            'icir': ic_series.mean() / ic_series.std() if ic_series.std() != 0 else 0,
            'ic_series': ic_series
        }
    
    # 4. 结果展示
    print("\n4. 结果汇总:")
    results_df = pd.DataFrame({
        'IC均值': [ic_results[name]['ic_mean'] for name in factor_results.keys()],
        'IC标准差': [ic_results[name]['ic_std'] for name in factor_results.keys()],
        'ICIR': [ic_results[name]['icir'] for name in factor_results.keys()]
    }, index=list(factor_results.keys()))
    
    print(results_df)
    
    # 5. 可视化
    print("\n5. 绘制分析图表...")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # IC均值柱状图
    results_df['IC均值'].plot(kind='bar', ax=ax1, title='不同Lambda值下的IC均值')
    ax1.axhline(0, color='red', linestyle='--')
    ax1.grid(True)
    
    # ICIR柱状图
    results_df['ICIR'].plot(kind='bar', ax=ax2, title='不同Lambda值下的ICIR')
    ax2.axhline(0, color='red', linestyle='--')
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    return factor_results, ic_results, results_df

# 运行主要分析（注释掉以避免在导入时执行）
# factor_results, ic_results, summary_df = main_analysis()

## 5. 使用示例

以下是如何使用上述代码的完整示例

In [ ]:
# 示例：快速开始
def quick_start_example():
    """快速开始示例"""
    
    # 设置参数
    start_date = '2020-01-01'
    end_date = '2023-12-31'
    
    print("开始动量因子分析...")
    
    try:
        # 运行完整分析
        results = main_analysis(start_date, end_date)
        
        if results:
            factor_results, ic_results, summary_df = results
            print("\n分析完成！")
            print("\n最佳因子（按ICIR排序）:")
            best_factor = summary_df.sort_values('ICIR', ascending=False).index[0]
            print(f"最佳因子: {best_factor}")
            print(f"ICIR: {summary_df.loc[best_factor, 'ICIR']:.4f}")
            
        else:
            print("分析失败，请检查数据源连接")
            
    except Exception as e:
        print(f"分析过程中出现错误: {e}")
        print("建议检查网络连接和tushare token设置")

# 运行示例（取消注释以执行）
# quick_start_example()

## 6. 高级功能

提供更多高级分析功能

In [ ]:
class AdvancedAnalysis:
    """高级分析功能"""
    
    @staticmethod
    def rolling_ic_analysis(factor_data: pd.DataFrame, price_data: pd.DataFrame, 
                           window: int = 60) -> pd.DataFrame:
        """滚动IC分析"""
        analyzer = FactorAnalyzer(factor_data, price_data)
        ic_series = analyzer.calculate_ic()
        
        rolling_ic = ic_series.rolling(window).mean()
        rolling_icir = ic_series.rolling(window).mean() / ic_series.rolling(window).std()
        
        return pd.DataFrame({
            'IC': ic_series,
            'Rolling_IC': rolling_ic,
            'Rolling_ICIR': rolling_icir
        })
    
    @staticmethod
    def factor_decay_analysis(factor_data: pd.DataFrame, price_data: pd.DataFrame,
                             periods: List[int] = [1, 5, 10, 20]) -> pd.DataFrame:
        """因子衰减分析"""
        decay_results = {}
        
        for period in periods:
            # 计算不同期间的收益率
            future_returns = price_data.pct_change(period).shift(-period)
            
            # 创建分析器
            temp_analyzer = FactorAnalyzer(factor_data, price_data)
            temp_analyzer.returns = future_returns
            
            # 计算IC
            ic_series = temp_analyzer.calculate_ic()
            decay_results[f'Period_{period}'] = {
                'IC_Mean': ic_series.mean(),
                'IC_Std': ic_series.std(),
                'ICIR': ic_series.mean() / ic_series.std() if ic_series.std() != 0 else 0
            }
        
        return pd.DataFrame(decay_results).T
    
    @staticmethod
    def sector_analysis(factor_data: pd.DataFrame, price_data: pd.DataFrame,
                       sector_mapping: Dict[str, str]) -> pd.DataFrame:
        """行业分析"""
        sector_results = {}
        
        for sector, stocks in sector_mapping.items():
            # 筛选行业内股票
            sector_stocks = [s for s in stocks if s in factor_data.columns]
            
            if len(sector_stocks) > 5:  # 至少5只股票
                sector_factor = factor_data[sector_stocks]
                sector_price = price_data[sector_stocks]
                
                analyzer = FactorAnalyzer(sector_factor, sector_price)
                ic_series = analyzer.calculate_ic()
                
                sector_results[sector] = {
                    'IC_Mean': ic_series.mean(),
                    'IC_Std': ic_series.std(),
                    'ICIR': ic_series.mean() / ic_series.std() if ic_series.std() != 0 else 0,
                    'Stock_Count': len(sector_stocks)
                }
        
        return pd.DataFrame(sector_results).T

# 示例使用
# advanced = AdvancedAnalysis()
# rolling_results = advanced.rolling_ic_analysis(factor_data, price_data)
# decay_results = advanced.factor_decay_analysis(factor_data, price_data)

## 7. 性能优化建议

为了提高代码执行效率，以下是一些优化建议：

In [ ]:
class PerformanceOptimizer:
    """性能优化工具"""
    
    @staticmethod
    def vectorized_momentum_calculation(returns: pd.DataFrame, window: int = 120) -> pd.DataFrame:
        """向量化动量计算（更快的实现）"""
        # 使用pandas的rolling函数进行向量化计算
        momentum = returns.rolling(window).sum()
        return momentum
    
    @staticmethod
    def parallel_ic_calculation(factor_data: pd.DataFrame, returns: pd.DataFrame,
                               n_jobs: int = -1) -> pd.Series:
        """并行IC计算"""
        from joblib import Parallel, delayed
        
        def calculate_single_ic(date):
            if date in returns.index:
                factor_values = factor_data.loc[date].dropna()
                return_values = returns.loc[date].dropna()
                
                common_stocks = factor_values.index.intersection(return_values.index)
                
                if len(common_stocks) > 10:
                    factor_aligned = factor_values[common_stocks]
                    return_aligned = return_values[common_stocks]
                    
                    ic, _ = stats.spearmanr(factor_aligned, return_aligned)
                    return ic
            return np.nan
        
        # 并行计算
        ic_values = Parallel(n_jobs=n_jobs)(
            delayed(calculate_single_ic)(date) for date in factor_data.index
        )
        
        return pd.Series(ic_values, index=factor_data.index)
    
    @staticmethod
    def memory_efficient_data_loading(stock_list: List[str], start_date: str, 
                                    end_date: str, batch_size: int = 50) -> pd.DataFrame:
        """内存高效的数据加载"""
        all_data = []
        
        # 分批加载数据
        for i in range(0, len(stock_list), batch_size):
            batch_stocks = stock_list[i:i+batch_size]
            print(f"加载第{i//batch_size + 1}批数据，共{len(batch_stocks)}只股票")
            
            batch_data = data_provider.get_price_data(batch_stocks, start_date, end_date)
            if not batch_data.empty:
                all_data.append(batch_data)
        
        if all_data:
            return pd.concat(all_data, axis=1)
        else:
            return pd.DataFrame()

# 使用示例
# optimizer = PerformanceOptimizer()
# fast_momentum = optimizer.vectorized_momentum_calculation(returns_data)
# fast_ic = optimizer.parallel_ic_calculation(factor_data, returns_data)

## 8. 总结与改进

### 主要改进点：

1. **数据源替换**：
   - 使用tushare替代jqdata获取基础数据
   - 添加akshare作为备用数据源
   - 实现了完整的数据获取封装

2. **代码结构优化**：
   - 采用面向对象设计，提高代码可维护性
   - 模块化设计，便于功能扩展
   - 添加详细的类型注解和文档字符串

3. **性能优化**：
   - 使用向量化计算替代循环
   - 支持并行计算提高效率
   - 内存高效的数据加载策略

4. **功能增强**：
   - 添加滚动IC分析
   - 因子衰减分析
   - 行业分析功能
   - 更丰富的可视化选项

5. **错误处理**：
   - 完善的异常处理机制
   - 数据质量检查
   - 友好的错误提示

### 使用建议：

1. 首次使用前请确保tushare token设置正确
2. 建议从小样本开始测试（如前100只股票）
3. 根据计算资源调整批处理大小和并行度
4. 定期保存中间结果以避免重复计算

### 扩展方向：

1. 添加更多因子构造方法
2. 集成机器学习模型
3. 实现实时因子监控
4. 添加风险模型和组合优化
5. 支持更多数据源（如Wind、Bloomberg等）